# SAE-Guided Causal Pruning - Colab Runner

Runs the same pipeline code as the GitHub repo, with the cache folder
pointed at Google Drive instead of Colab's local disk. Colab wipes local
disk on every disconnect, so without this, every stage would recompute
from scratch each session - Drive keeps SAE checkpoints, causal
features, masks, and results across sessions.

Run the cells in order, top to bottom.

In [ ]:
# Colab already has torch preinstalled - just add the rest
!pip install -q transformers datasets peft scikit-learn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Get the pipeline code

Option A (recommended): clone your GitHub repo - edit the URL below.

Option B: if you have not pushed to GitHub yet, upload the file
`sae_pruning_pipeline.zip` via the Colab file browser (folder icon,
left sidebar), then use Option B's cell instead of Option A's.

In [ ]:
# Option A - clone from GitHub (edit this URL to your own repo)
!git clone https://github.com/YOUR-USERNAME/YOUR-REPO-NAME.git
%cd YOUR-REPO-NAME/sae_pruning_pipeline

In [ ]:
# Option B - use this instead of Option A if you uploaded the zip directly
# !unzip -q sae_pruning_pipeline.zip
# %cd sae_pruning_pipeline

## Point the cache at Drive

This has to run before `import config`, since config.py reads the
cache root from this environment variable at import time.

In [ ]:
import os
os.environ['SAE_PRUNING_ROOT'] = '/content/drive/MyDrive/sae_pruning_cache'

import config
print('cache root:', config.ROOT)

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print('no GPU - go to Runtime > Change runtime type and select a GPU')

## Migrate existing data (skip this if starting fresh)

If you already have SAE checkpoints and causal features sitting in
/content/drive/MyDrive/sae_pruning from earlier work, this converts
them into the format this pipeline expects, so stage2 and stage3 get
real cache hits instead of silently retraining everything from scratch.
Safe to run even if you have nothing to migrate - it just does nothing.

In [ ]:
import migrate_old_data
migrate_old_data.run()

## Run the main comparison

Fresh baseline, then causal / magnitude / random pruning, all with no
fine-tuning, then the paired bootstrap comparisons. Every stage checks
the Drive cache first, so re-running this after a disconnect picks up
where it left off instead of starting over.

In [ ]:
import run_pipeline
run_pipeline.main()

## Cross-domain evaluation

Evaluates the same pruned models on Rotten Tomatoes and Yelp Polarity,
neither of which is used at any other stage of the pipeline. This tests
whether the causal advantage measured on IMDB also holds on data the
pruning decision was not derived from. Run this after the main
comparison above - it reuses the same cached pruning masks.

In [ ]:
import run_cross_domain
run_cross_domain.main()

## Optional: fine-tuning check

This is the supplementary check, not the main comparison - see the
README for why it is kept separate. Only run this if you specifically
want the recovery result.

In [ ]:
import stage7_finetune
stage7_finetune.run('causal', n_examples=4000)